# Metatlas2 sandbox workflow

End-to-end check of the targeted workflow on the sandbox data package: build a fresh database
(compounds → atlases → MS/MS refs), then project setup → RT alignment → auto-identification →
curation GUI → summary.

**Before you edit code:** you must be on a feature branch in your clone (`git switch -c <branch>`).
`metatlas2.sh --sandbox` refuses to start on `main`.

**Live edits:** your clone is mounted read-only into this container and comes first on `PYTHONPATH`.
Edit files on your machine, then **Kernel → Restart Kernel** and re-run the cells. Commit, push and
open the PR from your machine, not from this container.

**Outputs:** every `metatlas2.sh --sandbox` launch wipes the sandbox DB and outputs. Within one launch
you can do as many runs as you like (Part 2).

In [ ]:
import os
import logging
from pathlib import Path
from string import Template

import metatlas2.targeted.workflows as wfs
import metatlas2.shared_tools.load_tools as ldt
import metatlas2.shared_tools.logging_config as lcf
import metatlas2.shared_tools.database_interact as dbi
from metatlas2.shared_tools.paths import set_up_paths
from metatlas2.shared_tools.add_compounds_to_db import add_compounds_to_db
from metatlas2.shared_tools.add_atlases_to_db import add_atlases_to_db
from metatlas2.shared_tools.add_msms_refs_to_db import add_msms_refs_to_db

lcf.setup_logging(log_level=logging.INFO, log_to_stdout=True)
logger = lcf.get_logger("sandbox_workflow")

DATA_DIR = Path(os.environ["METATLAS_DATA_DIR"])           # ~/.metatlas2-sandbox on your machine
INPUTS = Path(os.environ["METATLAS2_REPO_DIR"]) / "sandbox" / "inputs"  # in your mounted clone
CONFIG_DIR = DATA_DIR / "configs"
MAIN_DB = DATA_DIR / "databases" / "main_db" / "metatlas.duckdb"
IMAGE_TAG = os.environ.get("METATLAS2_IMAGE_TAG", "latest")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
logger.info(f"Data dir: {DATA_DIR} | inputs: {INPUTS} | image tag: {IMAGE_TAG}")


def render(template_name: str, **values: str) -> Path:
    """Fill ${...} placeholders in sandbox/inputs/configs/<template_name> and write it to CONFIG_DIR."""
    text = Template((INPUTS / "configs" / template_name).read_text()).substitute(
        SANDBOX_INPUTS=str(INPUTS), **values
    )
    out = CONFIG_DIR / template_name
    out.write_text(text)
    return out

## Part 1 — Build the sandbox database (once per launch)

Safe to re-run: if the database already exists (from earlier in this launch), the build is skipped.

In [ ]:
if MAIN_DB.exists():
    logger.info(f"Sandbox DB already built this launch ({MAIN_DB}); skipping.")
else:
    logger.info("Adding compounds (creates the DB)...")
    add_compounds_to_db(str(INPUTS / "compounds.csv"))
    logger.info("Adding atlases...")
    add_atlases_to_db(str(render("atlases_config.yaml")))
    logger.info("Adding MS/MS reference spectra...")
    add_msms_refs_to_db(str(render("msms_refs_config.yaml")))
    logger.info("Sandbox DB built.")

In [ ]:
# Look up the UIDs add-atlases assigned and write them into the analysis config.
atlas_uids = {}
for _, row in dbi.query_atlases_metadata(str(MAIN_DB)).iterrows():
    key = "_".join(["UID", row["chromatography"], row["polarity"], row["atlas_category"], row["atlas_name"]]).upper()
    atlas_uids[key] = row["atlas_uid"]
logger.info(f"Atlas UIDs: {atlas_uids}")
ANALYSIS_CONFIG = render("analysis_config.yaml", **atlas_uids)
logger.info(f"Analysis config: {ANALYSIS_CONFIG}")

## Part 2 — Run the targeted workflow (repeatable)

To do another run in this launch, change `rt_alignment_number` and/or `analysis_number`
(e.g. after editing code and restarting the kernel) and re-run from here. Earlier runs are kept
until the next `metatlas2.sh --sandbox` launch.

In [ ]:
project_name = "20260101_JGI_XX_000000_SANDBOX_test_EXP000_HILICZ_TESTXXXX"
rt_alignment_number = 0
analysis_number = 0

In [ ]:
config = ldt.load_metatlas2_config(str(ANALYSIS_CONFIG))
paths = set_up_paths(
    config=config,
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)
lcf.setup_logging(log_level=logging.INFO, log_file=paths["log_path"], log_to_stdout=True)
logger = lcf.get_logger("sandbox_workflow")
logger.info(f"Run: {project_name} RTA{rt_alignment_number} TGA{analysis_number}")

In [ ]:
wfs.run_project_setup(
    project_name=project_name,
    config=config,
    paths=paths,
    overwrite_existing=True,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)

In [ ]:
wfs.run_rt_alignment(
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
)

In [ ]:
wfs.run_auto_identification(
    project_name=project_name,
    rt_alignment_number=rt_alignment_number,
    analysis_number=analysis_number,
    image_tag=IMAGE_TAG,
)
logger.info(f"Curation notebooks: {paths['project_directory']}")

## Part 3 — Curation GUI and summary

Open the generated notebook(s) in the project directory printed above (in the file browser:
`projects/targeted_outputs/jgi/<user>/<project>/RTA<N>/TGA<M>/`), run the GUI cell, click
**Open Dash App** (http://localhost:8050/), curate, then run the summary cell.